# agentic_alpha_rgevd_search_v1

Diagnostic R-G-E-V-D search card for SOXL. This notebook reads external knowledge-source seeds, retrieves qlib SOXL/SOXX/VIX data, generates volatility/trend/autocorrelation candidates, evaluates them against the current production qlib paper-sim baseline, validates the `MaxDD < 20%` hard gate, and distills whether any candidate is promotable.

This is a diagnostic search card, not a validated promotion card.

In [1]:

from __future__ import annotations

import json
import sys
from pathlib import Path

import os
ROOT_CANDIDATES = [Path.cwd(), Path(os.environ.get('PWD', '.')), Path('/Users/rziiiii/dev/workspace')]
ROOT = next(
    path
    for base in ROOT_CANDIDATES
    for path in [base, *base.parents]
    if (path / 'AGENTS.md').exists() and (path / 'SPEC.md').exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from qlib.workflow import R

from src.workflow import (
    AgenticAlphaBacktestMetric,
    AgenticAlphaKnowledgeSource,
    AgenticAlphaSearchCandidate,
    build_agentic_alpha_hypotheses,
    distill_agentic_alpha_search,
    load_agentic_alpha_backtest_metric,
)

CARD = ROOT / "research" / "agentic_alpha_rgevd_search_v1"
EXPERIMENT = "agentic_alpha_rgevd_search_v1"
PROVIDER_URI = (ROOT / "data" / "qlib" / "soxl").resolve()
RECORDER_URI = (CARD / "qlib_recorder").resolve()
START = "2019-01-02"
END = "2026-06-09"
MAX_DRAWDOWN_FLOOR = -20.0

CARD.mkdir(parents=True, exist_ok=True)
RECORDER_URI.mkdir(parents=True, exist_ok=True)

qlib.init(provider_uri=PROVIDER_URI.as_posix(), region="us", kernels=1)

baseline = load_agentic_alpha_backtest_metric(ROOT / "production" / "qlib_paper_sim_v1", window="test")

sources = [
    AgenticAlphaKnowledgeSource(
        source_type="github",
        title="agentic-alpha R-G-E-V-D workflow",
        url="https://github.com/JacobDu/agentic-alpha",
        extracted_signal="Use Research/Retrieve/Generate/Evaluate/Validate/Distill to keep factor ideas evidence-backed.",
        tags=("workflow", "factor"),
    ),
    AgenticAlphaKnowledgeSource(
        source_type="paper",
        title="Conditional volatility targeting",
        url="https://alphaarchitect.com/conditional-volatility-targeting/",
        extracted_signal="Conditional volatility targeting can reduce drawdowns and tail risk.",
        tags=("volatility", "risk"),
    ),
    AgenticAlphaKnowledgeSource(
        source_type="twitter",
        title="SOXL leveraged ETF rebalancing loop discussion",
        url="https://x.com/ces921/highlights",
        extracted_signal="Levered ETF rebalancing loop can amplify regime risk in SOXL.",
        tags=("flow", "regime"),
    ),
    AgenticAlphaKnowledgeSource(
        source_type="arxiv",
        title="Compounding Effects in Leveraged ETFs",
        url="https://arxiv.org/html/2504.20116v1",
        extracted_signal="LETF compounding depends on return autocorrelation, volatility clustering, and regime persistence.",
        tags=("autocorrelation", "volatility"),
    ),
    AgenticAlphaKnowledgeSource(
        source_type="arxiv",
        title="Single-Asset Adaptive Leveraged Volatility Control",
        url="https://arxiv.org/html/2603.01298v2",
        extracted_signal="Adaptive target-volatility control uses interpretable feedback and leverage limits.",
        tags=("target_volatility", "control"),
    ),
]
hypotheses = build_agentic_alpha_hypotheses(sources)

soxl = D.features(["SOXL"], ["$close", "$vix"], start_time=START, end_time=END).reset_index()
soxx = D.features(["SOXX"], ["$close"], start_time=START, end_time=END).reset_index()
soxl["date"] = pd.to_datetime(soxl["datetime"])
soxx["date"] = pd.to_datetime(soxx["datetime"])
frame = (
    soxl[["date", "$close", "$vix"]]
    .rename(columns={"$close": "soxl_close", "$vix": "vix"})
    .merge(soxx[["date", "$close"]].rename(columns={"$close": "soxx_close"}), on="date", how="left")
    .sort_values("date")
    .reset_index(drop=True)
)
frame["soxl_ret"] = frame["soxl_close"].pct_change().fillna(0.0)
frame["soxx_ret"] = frame["soxx_close"].pct_change().fillna(0.0)


def path_metrics(weight: pd.Series) -> dict[str, float]:
    weight = pd.Series(weight, index=frame.index).shift(1).fillna(0.0).clip(0.0, 1.0)
    turnover = weight.diff().abs().fillna(weight.abs())
    strategy_return = weight * frame["soxl_ret"] - turnover * 0.0003
    nav = (1.0 + strategy_return).cumprod()
    drawdown = nav / nav.cummax() - 1.0
    benchmark_nav = (1.0 + frame["soxl_ret"]).cumprod()
    return {
        "return_pct": float((nav.iloc[-1] - 1.0) * 100.0),
        "max_drawdown_pct": float(drawdown.min() * 100.0),
        "benchmark_return_pct": float((benchmark_nav.iloc[-1] - 1.0) * 100.0),
        "turnover": float(turnover.sum()),
        "mean_exposure_pct": float(weight.mean() * 100.0),
    }

rows = []
for fast in [5, 10, 20, 50]:
    for slow in [50, 100, 150, 200]:
        if fast >= slow:
            continue
        for vix_max in [22, 25, 28, 30, 35, 40]:
            for target_vol in [None, 0.035, 0.045, 0.055, 0.065]:
                for crash_vix in [28, 35, 45]:
                    trend = frame["soxx_close"].rolling(fast).mean() > frame["soxx_close"].rolling(slow).mean()
                    vix_gate = frame["vix"] <= vix_max
                    weight = (trend & vix_gate).astype(float)
                    if target_vol is not None:
                        realized = frame["soxl_ret"].rolling(20).std() * np.sqrt(252)
                        vol_weight = (target_vol / realized.replace(0, np.nan)).clip(0.0, 1.0).fillna(0.0)
                        weight = weight * vol_weight
                    weight = weight.where(frame["vix"] <= crash_vix, 0.0)
                    row = path_metrics(weight)
                    row.update(
                        {
                            "fast": fast,
                            "slow": slow,
                            "vix_max": vix_max,
                            "target_vol": target_vol if target_vol is not None else "none",
                            "crash_vix": crash_vix,
                        }
                    )
                    rows.append(row)

grid = pd.DataFrame(rows)
grid["passes_drawdown_gate"] = grid["max_drawdown_pct"] >= MAX_DRAWDOWN_FLOOR
grid["beats_baseline_return"] = grid["return_pct"] > baseline.return_pct
high_return = grid.sort_values("return_pct", ascending=False).iloc[0]
dd20_pool = grid.loc[grid["passes_drawdown_gate"]]
risk_control = dd20_pool.sort_values("return_pct", ascending=False).iloc[0] if not dd20_pool.empty else grid.sort_values("max_drawdown_pct", ascending=False).iloc[0]

candidates = [
    AgenticAlphaSearchCandidate(
        name="vol_target_trend_grid_high_return",
        hypothesis_id=hypotheses[0].hypothesis_id,
        source_urls=hypotheses[0].source_urls,
        metric=AgenticAlphaBacktestMetric(
            name="vol_target_trend_grid_high_return",
            return_pct=float(high_return["return_pct"]),
            max_drawdown_pct=float(high_return["max_drawdown_pct"]),
            benchmark_return_pct=float(high_return["benchmark_return_pct"]),
            recorder_id="diagnostic_grid_no_portana_high_return",
        ),
    ),
    AgenticAlphaSearchCandidate(
        name="vol_target_trend_grid_dd20",
        hypothesis_id=hypotheses[0].hypothesis_id,
        source_urls=hypotheses[0].source_urls,
        metric=AgenticAlphaBacktestMetric(
            name="vol_target_trend_grid_dd20",
            return_pct=float(risk_control["return_pct"]),
            max_drawdown_pct=float(risk_control["max_drawdown_pct"]),
            benchmark_return_pct=float(risk_control["benchmark_return_pct"]),
            recorder_id="diagnostic_grid_no_portana_dd20",
        ),
    ),
]

report = distill_agentic_alpha_search(
    sources=sources,
    hypotheses=hypotheses,
    candidates=candidates,
    baseline=baseline,
    max_drawdown_floor=MAX_DRAWDOWN_FLOOR,
    require_return_above_baseline=True,
)

external_sources = pd.DataFrame(
    [
        {
            "source_type": source.source_type,
            "title": source.title,
            "url": source.url,
            "extracted_signal": source.extracted_signal,
            "tags": ";".join(source.tags),
        }
        for source in sources
    ]
)
hypothesis_rows = pd.DataFrame(
    [
        {
            "hypothesis_id": hypothesis.hypothesis_id,
            "stage": hypothesis.stage,
            "theme": hypothesis.theme,
            "candidate_logic": hypothesis.candidate_logic,
            "source_types": ";".join(hypothesis.source_types),
            "source_urls": ";".join(hypothesis.source_urls),
        }
        for hypothesis in hypotheses
    ]
)

R.set_uri(RECORDER_URI.as_posix())
with R.start(experiment_name=EXPERIMENT):
    recorder = R.get_recorder()
    recorder_id = recorder.id
    report.summary["recorder_id"] = recorder_id
    report.summary["diagnostic_grid"] = {
        "rows": int(len(grid)),
        "passes_both_count": int((grid["passes_drawdown_gate"] & grid["beats_baseline_return"]).sum()),
        "best_return_pct": float(high_return["return_pct"]),
        "best_return_max_drawdown_pct": float(high_return["max_drawdown_pct"]),
        "best_dd20_return_pct": float(risk_control["return_pct"]),
        "best_dd20_max_drawdown_pct": float(risk_control["max_drawdown_pct"]),
    }
    sidecar_paths = report.write_sidecars(CARD)
    external_path = CARD / "external_sources.csv"
    hypotheses_path = CARD / "hypotheses.csv"
    external_sources.to_csv(external_path, index=False)
    hypothesis_rows.to_csv(hypotheses_path, index=False)
    sidecar_paths["external_sources"] = external_path
    sidecar_paths["hypotheses"] = hypotheses_path

    R.log_params(
        provider_uri=PROVIDER_URI.as_posix(),
        start=START,
        end=END,
        baseline=baseline.name,
        max_drawdown_floor=MAX_DRAWDOWN_FLOOR,
        candidate_count=len(candidates),
        diagnostic_grid_rows=len(grid),
    )
    for path in sidecar_paths.values():
        recorder.log_artifact(path.as_posix(), artifact_path="agentic_alpha_rgevd")
    recorder_local_dir = Path(recorder.get_local_dir())

for transient_name in ["code_cached.txt", "code_diff.txt", "code_status.txt"]:
    transient_path = recorder_local_dir / "artifacts" / transient_name
    if transient_path.exists():
        transient_path.unlink()

recorder_artifacts = sorted(
    path.relative_to(recorder_local_dir).as_posix() for path in recorder_local_dir.rglob("*") if path.is_file()
)
summary = {
    "experiment": EXPERIMENT,
    "card_type": "diagnostic_search",
    "workflow_config": "workflow.yaml",
    "provider_uri": PROVIDER_URI.as_posix(),
    "recorder_uri": RECORDER_URI.as_posix(),
    "recorder_id": recorder_id,
    "recorder_local_dir": recorder_local_dir.as_posix(),
    "recorder_artifacts": recorder_artifacts,
    "market": "all",
    "region": "us",
    "as_of": END,
    "data_range": [START, END],
    "qlib_native_api": ["qlib.init", "qlib.data.D.features", "qlib.workflow.R.start"],
    "qlib_docs": {
        "data": "https://qlib.readthedocs.io/en/latest/component/data.html",
        "workflow": "https://qlib.readthedocs.io/en/latest/component/workflow.html",
    },
    "external_source_counts": report.summary["external_source_counts"],
    "baseline": report.summary["baseline"],
    "goal_gate": report.summary["gates"],
    "search_report": report.summary,
    "promotion_decision": "not_promoted" if not report.summary["goal_achieved"] else "promote_candidate",
    "sidecars": {
        "candidate_metrics": "agentic_alpha_candidate_metrics.csv",
        "external_sources": "external_sources.csv",
        "hypotheses": "hypotheses.csv",
        "search_summary": "agentic_alpha_search_summary.json",
    },
}
summary["artifact_policy"] = {
    "policy": "slim_card_recorder_first",
    "primary_artifact": f"{EXPERIMENT}.ipynb",
    "machine_entrypoint": "summary.json",
    "workflow_config": "workflow.yaml",
    "workflow_config": "workflow.yaml",
    "kept_sidecars": sorted(summary["sidecars"].values()),
    "reproducible_from_recorder": True,
    "recorder_only_artifacts": {
        "predictions": "artifacts/pred.pkl",
        "labels": "artifacts/label.pkl",
        "all_model_predictions": "artifacts/model_selection/all_model_predictions.csv",
        "signals": "artifacts/signals.pkl",
        "portfolio_report": "artifacts/portfolio_analysis/report_normal_1day.pkl",
        "positions": "artifacts/portfolio_analysis/positions_normal_1day.pkl",
        "risk_analysis": "artifacts/portfolio_analysis/port_analysis_1day.pkl",
    },
}
(CARD / "summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True), encoding="utf-8")
conclusion = f"""# 中文结论

- 做了什么：用 R-G-E-V-D 流程把 GitHub、论文、Twitter/X、arXiv 外部知识转成 SOXL 候选假设，并通过 qlib.init 与 D.features 读取 SOXL/SOXX/VIX 数据做诊断搜索。
- 核心指标：当前生产基准 {baseline.name} 的 test return_pct={baseline.return_pct:.4f}，max_drawdown_pct={baseline.max_drawdown_pct:.4f}；搜索网格共 {len(grid)} 个路径，最高收益候选 return_pct={float(high_return['return_pct']):.4f}、max_drawdown_pct={float(high_return['max_drawdown_pct']):.4f}，最佳 20% 回撤门槛候选 return_pct={float(risk_control['return_pct']):.4f}、max_drawdown_pct={float(risk_control['max_drawdown_pct']):.4f}。
- regime/window 差异：候选来自趋势、VIX 风险门、目标波动率和高 VIX 去风险组合；高收益候选仍承受深回撤，低回撤候选收益远低于生产基准。
- 是否晋升：不晋升；未找到同时满足收益超过现有策略且 MaxDD < 20% 的候选。
- 风险：这是 diagnostic search，不是 PortAnaRecord 晋升卡；候选路径用 qlib 数据诊断计算，后续若出现过门槛候选必须升级为 qlib recorder/PortAnaRecord validated card。
- 下一步：扩大到模型化 regime selector、intraday T+1 特征和 walk-forward 搜索，而不是继续堆叠静态规则。
- qlib recorder id：{recorder_id}
"""
(CARD / "conclusion.md").write_text(conclusion, encoding="utf-8")
print(f"recorder_id={recorder_id}")
print(f"goal_achieved={report.summary['goal_achieved']}")
print(f"passes_both_count={report.summary['diagnostic_grid']['passes_both_count']}")
print(f"candidate_metrics={sidecar_paths['candidate_metrics'].name}")


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[26794:MainThread](2026-06-23 15:27:17,080) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[26794:MainThread](2026-06-23 15:27:17,082) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[26794:MainThread](2026-06-23 15:27:17,083) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl')}


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
[26794:MainThread](2026-06-23 15:27:18,346) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name agentic_alpha_rgevd_search_v1.


[26794:MainThread](2026-06-23 15:27:18,348) INFO - qlib.workflow - [exp.py:258] - Experiment 580566699392312645 starts running ...


[26794:MainThread](2026-06-23 15:27:18,465) INFO - qlib.workflow - [recorder.py:345] - Recorder 5833c5c24a6841929b9aac55bdc7c1d1 starts running under Experiment 580566699392312645 ...


[26794:MainThread](2026-06-23 15:27:18,536) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | waiting `async_log` Done


recorder_id=5833c5c24a6841929b9aac55bdc7c1d1
goal_achieved=False
passes_both_count=0
candidate_metrics=agentic_alpha_candidate_metrics.csv


## 中文结论

本 notebook 执行后会在 `conclusion.md` 写入最终中文结论。当前卡是 diagnostic search：只有当候选同时超过现有 production baseline 且 `max_drawdown_pct >= -20.0` 时才允许进入后续 validated qlib/PortAnaRecord 晋升卡。